In [21]:
import os
import shutil
from pathlib import Path

import pdf2image
import pytesseract

filename = "limburg.pdf"

# 1) Intentar usar POPPLER_PATH si existe
# 2) Si no, buscar en rutas tipicas de Windows
# 3) Detectar instalacion con winget
# 4) Si existe pdfinfo en PATH, usar su carpeta como poppler_path
candidate_paths = []
if os.getenv("POPPLER_PATH"):
    candidate_paths.append(Path(os.getenv("POPPLER_PATH")))

candidate_paths.extend([
    Path(r"C:\poppler\Library\bin"),
    Path(r"C:\Program Files\poppler\Library\bin"),
    Path.home() / r"scoop\apps\poppler\current\Library\bin",
])

winget_packages = Path.home() / r"AppData\Local\Microsoft\WinGet\Packages"
if winget_packages.exists():
    for pkg_dir in winget_packages.glob("oschwartz10612.Poppler*"):
        for pdfinfo_file in pkg_dir.rglob("pdfinfo.exe"):
            candidate_paths.append(pdfinfo_file.parent)

pdfinfo_exe = shutil.which("pdfinfo")
if pdfinfo_exe:
    candidate_paths.append(Path(pdfinfo_exe).parent)

poppler_bin = next((str(p) for p in candidate_paths if p.exists()), None)

if poppler_bin is None:
    raise FileNotFoundError(
        "No se encontro Poppler. Instala Poppler y define POPPLER_PATH con la ruta a ...\\Library\\bin."
    )

paginas = pdf2image.convert_from_path(filename, poppler_path=poppler_bin)

for num_pagina, imagen in enumerate(paginas, start=1):
    texto = pytesseract.image_to_string(imagen)
    print(f"Pagina {num_pagina}:\n{texto}\n{'-' * 40}\n")

TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.